# Footsquare RAG: multiple-choice QA on the rules of a new sport

Footsquare is an invented sport (circular pitch, raised baskets, keeper zones), so a language model
cannot rely on prior knowledge: every answer must come from the rulebook. This notebook builds a
Retrieval-Augmented Generation (RAG) pipeline that answers 25 multiple-choice questions about its rules.

**Pipeline**
1. **Index**: split the rulebook into chunks and embed them with `Qwen/Qwen3-Embedding-0.6B`.
2. **Retrieve**: embed each question and select the 7 most similar chunks by cosine similarity.
3. **Generate**: give the retrieved rules, the question and the options to `unsloth/gemma-3-1B-it`
   and extract the chosen letter with greedy decoding.
4. **Judge**: use an LLM-as-a-judge to estimate grounding and context reliability (no labels are available).
5. **Export**: write the predictions to a CSV for the Codabench evaluation.

**Result**: 0.760 accuracy (19/25) on Codabench.


## Setup

Import the project modules, define the data and model-cache folders and download the knowledge base
(`rules.json`, 49 rules) and the questions (`questions_mcq_student.json`, 25 questions) if they are not
already present. On Colab, first clone the repository so that `src/` is importable.

In [ ]:
# On Colab, uncomment:
# !git clone https://github.com/Gabri03ele/footsquare-rag.git
# %cd footsquare-rag

import sys
from pathlib import Path
from collections import Counter

if "google.colab" in sys.modules:
    import os
    if not os.path.exists("/content/footsquare-rag"):
        !git clone https://github.com/Gabri03ele/footsquare-rag.git /content/footsquare-rag
    else:
        !git -C /content/footsquare-rag pull
    %cd /content/footsquare-rag

from pathlib import Path
from src.retrieve import augment_query, retrieve_top_n
from collections import Counter
from src.data import download_data, load_rules, load_questions
from src.index import load_embedder, build_index
from src.generate import load_generator, answer_question
from src.judge import judge_result, compute_scores
from src.export import write_predictions, prediction_distribution


# make src/ importable whether the notebook runs from the repo root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.append(str(ROOT))

DATA_DIR = ROOT / "data"
CACHE_DIR = ROOT / "models_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

download_data(DATA_DIR)
rules = load_rules(DATA_DIR)
questions = load_questions(DATA_DIR)

print("Rules:", len(rules))
print("Questions:", len(questions))

## Build the index

Each rule is split on sentence boundaries into chunks of at most 200 words, with a 40-word overlap
between consecutive chunks, and every chunk is embedded with `Qwen/Qwen3-Embedding-0.6B`.
Embeddings are L2-normalized, so cosine similarity reduces to a dot product at retrieval time.

The rules are short (one or two sentences each), so in practice every rule becomes a single chunk.
The chunking is kept so that the pipeline also works with a longer rulebook.

In [ ]:
embedder = load_embedder(CACHE_DIR)
KB_CHUNKS = build_index(rules, embedder, max_length=200, overlap=40)

print("KB chunks:", len(KB_CHUNKS))
print("Example:", KB_CHUNKS[0]["id"], "->", KB_CHUNKS[0]["text"])

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

KB chunks: 49


## Retrieval

For each question, the query is first augmented with the numbers and keywords found in the question
and its options, so that chunks containing the same values (e.g. "5 m", "2 points") score higher.
The augmented query is embedded with the same model used for the index, and the top-n chunks are
selected by cosine similarity (a dot product, since all embeddings are normalized).

The pipeline uses n = 7. The cell below shows the top 3 chunks retrieved for the first question.

In [ ]:
sample = questions[0]
options_text = "\n".join(f"{opt['id']}) {opt['text']}" for opt in sample["options"])
query = augment_query(sample["question"], options_text)

top = retrieve_top_n(query, KB_CHUNKS, embedder, n=3)

print("Question:", sample["question"])
print("Augmented query:", query)
for item in top:
    print(f"- similarity={item['similarity']:.3f}  {item['text'][:160]}")

Question: Which statement correctly describes the Footsquare pitch boundary?
- similarity=0.766
  The pitch is circular with a clearly marked boundary line. The boundary line is part of the field of play....
- similarity=0.689
  Footsquare is a team sport played on a rounded pitch with raised baskets instead of goals. The game emphasizes continuous play, precise passing, and 360-degree ...
- similarity=0.672
  Footsquare uses a zone rule instead of offside. Attacking players may not receive the ball inside the opponent's keeper zone....


## Generation

For each question, the retrieved rules are numbered and inserted into a prompt together with the
question and its options. `unsloth/gemma-3-1B-it` answers with greedy decoding (deterministic output),
and the chosen letter is extracted from its reply with three fallbacks:
an explicit "Final answer: X", then "answer is X", then the last standalone letter A–E.
If no letter is found the answer is marked as `z`, so failed extractions remain visible.

In [ ]:
tokenizer_gemma, model_gemma = load_generator(CACHE_DIR)

RESULTS = []
for q in questions:
    result = answer_question(q, KB_CHUNKS, embedder, tokenizer_gemma, model_gemma, n=7)
    RESULTS.append(result)

    print(f"{result['id']}: {result['query']}")
    for chunk in result["retrieved"]:
        print(f"   similarity={chunk['similarity']:.3f}  {chunk['text'][:120]}")
    print(f"   -> answer: {result['answer']}\n")

answer_dist = Counter(r["answer"] for r in RESULTS)
print("Answer distribution:", dict(answer_dist))
print("Failed extractions:", answer_dist.get("z", 0))

config.json:   0%|          | 0.00/902 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

... (3 outputs left out)

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

Query: Which statement correctly describes the Footsquare pitch boundary?
Similarity=0.750, text=The pitch is circular with a clearly marked boundary line. The boundary line is part of the field of play.
Similarity=0.660, text=Footsquare is a team sport played on a rounded pitch with raised baskets instead of goals. The game emphasizes continuous play, precise passing, and 360-degree spatial awareness.
Similarity=0.659, text=Footsquare uses a zone rule instead of offside. Attacking players may not receive the ball inside the opponent's keeper zone.
Similarity=0.590, text=Throw-in: taken from the nearest quadrant line when the ball leaves the field.
Similarity=0.579, text=Four quadrant lines divide the pitch into equal arcs. These lines are used for restarts and zone-based rules.
Similarity=0.573, text=A foul results in a free shot from the nearest quadrant line.
Similarity=0.572, text=If the ball crosses the boundary line, play restarts with a throw-in from the nearest quadrant line.


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Final Answer: D


Query: What is the radius of the central circle used for kickoff?
Similarity=0.818, text=A central circle with a radius of 6 m marks the kickoff area. A surrounding ring (the control ring) extends from 6 m to 12 m.
Similarity=0.676, text=The match begins with a center kick from the central circle. Opponents must remain outside the control ring.
Similarity=0.598, text=Center kick: starts each period and follows every score.
Similarity=0.570, text=After a score, the team that conceded restarts with a center kick.
Similarity=0.538, text=The keeper may handle the ball only within the keeper zone, a 4 m radius circle beneath the basket.
Similarity=0.519, text=A dropped ball is used for neutral stoppages and is taken at the nearest quadrant line.
Similarity=0.516, text=The basket has a rim diameter of 0.9 m with a shallow net. A score counts only when the ball fully passes through the rim from above.
Final Answer: A


Query: Which statement about the control ring is correct

## LLM-as-a-judge

No labels are available, so the same Gemma model is used as a judge to estimate the quality of the
pipeline with two yes/no questions per answer:
1. **Grounding**: is the selected answer supported by the retrieved rules?
2. **Context reliability**: do the retrieved rules contain the information needed to answer?

The judge is a 1B model that also generated the answers, so these numbers are a rough indication,
not an accuracy measure.

In [ ]:
JUDGMENTS = [judge_result(r, tokenizer_gemma, model_gemma) for r in RESULTS]
scores = compute_scores(JUDGMENTS)

print(f"Grounding rate: {scores['grounding_rate']:.1%}")
print(f"Context reliability rate: {scores['reliability_rate']:.1%}")
print(f"Both good: {scores['both_good_rate']:.1%}")

Judged: 25
Sample: {'query': 'Which statement correctly describes the Footsquare pitch boundary?', 'grounding': 'No', 'context_reliability': 'Yes'}

Performance Metrics:
  Grounding Rate: 76.0% (19/25)
  Context Reliability Rate: 96.0% (24/25)
  Both Good Rate: 76.0% (19/25)
Overall score: {'grounding_rate': 0.76, 'reliability_rate': 0.96, 'both_good_rate': 0.76, 'total_queries': 25}


## Answering

Each option is scored in two ways and the scores are averaged with equal weight:
1. **Language model**: `unsloth/gemma-3-1B-it` reads the retrieved rules, the question and the options,
   and the probability of each letter after "Final answer:" is taken from its output. This is repeated
   with the options in five rotated orders and the probabilities are averaged, which removes the model's
   bias against the last option.
2. **Option–rule similarity**: each option is embedded with its question and compared with the
   retrieved rules.

The option with the highest combined score is the answer.

In [ ]:
from src.evaluate import load_gold, print_report

GOLD = load_gold(ROOT / "eval" / "gold_labels.json")
print_report({r["id"]: r["answer"] for r in RESULTS}, GOLD, name="hybrid")

results_hy = [answer_question(q, KB_CHUNKS, embedder, tokenizer_gemma, model_gemma, n=7, method="hybrid")
              for q in questions]

def best(scores):
    return max(scores, key=scores.get)

print_report({r["id"]: best(r["llm_scores"]) for r in results_hy}, GOLD, name="permute (LLM only)")
print_report({r["id"]: best(r["emb_scores"]) for r in results_hy}, GOLD, name="embedding only")
print_report({r["id"]: r["answer"] for r in results_hy}, GOLD, name="hybrid 50/50")
print("hybrid distribution:", dict(Counter(r["answer"] for r in results_hy)))